# 04 — Buying Group Analysis & Coverage Gaps

This is the strategic layer on top of the propensity model. The model tells us *which accounts are likely to close*. The buying group analysis tells us *why* — and more importantly, *what's missing* at accounts that could close but haven't yet.

**The business question:** Which accounts have incomplete buying groups, and where should marketing invest in enrichment to fill the gaps?

This is the analysis that directly mirrors the $50M pipeline work at Atlassian — identifying coverage gaps across accounts and translating them into enrichment strategies with estimated pipeline value.

**Approach:**
1. Score every account's buying group completeness (0-100)
2. Analyze how completeness drives deal outcomes
3. Identify the specific gaps — missing roles, missing seniority, missing functions
4. Estimate the pipeline value of filling those gaps
5. Generate actionable enrichment recommendations

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 100

from src.buying_groups import (
    score_buying_group_completeness,
    identify_coverage_gaps,
    completeness_vs_win_rate,
    estimate_enrichment_pipeline,
)

# Load data
accounts = pd.read_csv("../data/accounts.csv")
contacts = pd.read_csv("../data/contacts.csv")
opportunities = pd.read_csv("../data/opportunities.csv")
contact_opp = pd.read_csv("../data/contact_opportunity.csv")

print(f"Accounts:      {len(accounts):,}")
print(f"Contacts:      {len(contacts):,}")
print(f"Opportunities: {len(opportunities):,}")
print(f"Contact roles: {len(contact_opp):,}")

## 1. Buying Group Completeness Scoring

Every account with deal activity gets a completeness score from 0-100, based on four dimensions (25 points each):

| Dimension | What it measures | 25 points if... |
|-----------|-----------------|-----------------|
| **Role coverage** | Are the key buying roles present? | All 4 roles (Champion, Decision Maker, Evaluator, Influencer) |
| **Seniority mix** | Is there exec-level AND mid-level engagement? | Has both VP+ and Director/Manager contacts |
| **Function diversity** | Are multiple departments involved? | 3+ distinct job functions |
| **Tech + Business** | Are both builders and buyers at the table? | Has technical AND business function contacts |

In [ ]:
completeness = score_buying_group_completeness(accounts, contacts, contact_opp, opportunities)

print(f"Accounts scored: {len(completeness)}")
print(f"Accounts with deal activity: {(completeness['contact_count'] > 0).sum()}")
print(f"Accounts with no deals: {(completeness['contact_count'] == 0).sum()}")
print(f"\nCompleteness score distribution (accounts with deals):")
active = completeness[completeness["contact_count"] > 0]
print(active["completeness_score"].describe().to_string())

In [ ]:
# Completeness distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall distribution
axes[0].hist(active["completeness_score"], bins=20, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Completeness Score")
axes[0].set_ylabel("Accounts")
axes[0].set_title("Buying Group Completeness Distribution")
axes[0].axvline(x=active["completeness_score"].median(), color="red", linestyle="--",
                alpha=0.7, label=f'Median: {active["completeness_score"].median():.0f}')
axes[0].legend()

# Sub-score breakdown
sub_scores = active[["role_coverage_score", "seniority_mix_score",
                      "function_diversity_score", "tech_business_score"]].mean()
sub_labels = ["Role\nCoverage", "Seniority\nMix", "Function\nDiversity", "Tech +\nBusiness"]
colors = sns.color_palette("muted", 4)
bars = axes[1].bar(sub_labels, sub_scores.values, color=colors, edgecolor="white")
axes[1].set_ylabel("Average Score (out of 25)")
axes[1].set_title("Average Sub-Score by Dimension")
axes[1].set_ylim(0, 25)
for bar, val in zip(bars, sub_scores.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f"{val:.1f}", ha="center", fontsize=11)

plt.tight_layout()
plt.show()

## 2. Completeness vs. Deal Outcomes

The central question: does buying group completeness actually predict whether a deal closes? If it does, then filling gaps should improve win rates.

In [ ]:
tier_summary = completeness_vs_win_rate(completeness, opportunities)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Win rate by tier
bars = axes[0].bar(range(len(tier_summary)), tier_summary["win_rate"],
                   color=sns.color_palette("muted"), edgecolor="white")
axes[0].set_xticks(range(len(tier_summary)))
axes[0].set_xticklabels(tier_summary["completeness_tier"])
axes[0].set_ylabel("Win Rate")
axes[0].set_title("Win Rate by Buying Group Completeness")
axes[0].set_ylim(0, 0.7)
for bar, (_, row) in zip(bars, tier_summary.iterrows()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{row["win_rate"]:.0%}\n(n={row["account_count"]:.0f})',
                 ha="center", fontsize=9)

# Pipeline value by tier
bars2 = axes[1].bar(range(len(tier_summary)), tier_summary["total_pipeline"],
                    color=sns.color_palette("muted"), edgecolor="white")
axes[1].set_xticks(range(len(tier_summary)))
axes[1].set_xticklabels(tier_summary["completeness_tier"])
axes[1].set_ylabel("Total Pipeline ($)")
axes[1].set_title("Pipeline Value by Buying Group Completeness")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f"${x/1e6:.1f}M"))
for bar, (_, row) in zip(bars2, tier_summary.iterrows()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + tier_summary["total_pipeline"].max() * 0.02,
                 f'${row["total_pipeline"]/1e6:.1f}M', ha="center", fontsize=9)

plt.tight_layout()
plt.show()

print("Completeness tier summary:")
print(tier_summary.to_string(index=False, float_format="{:.2f}".format))

## 3. What's Missing? — Gap Analysis

Now the actionable part. For accounts with incomplete buying groups, what *specifically* are they missing? This is the input to marketing's enrichment strategy — "go find Champions at these 50 accounts" or "we need VP-level contacts at these Enterprise accounts."

In [ ]:
# Identify accounts with gaps (completeness 0-75, meaning at least one dimension is weak)
gaps = identify_coverage_gaps(completeness, accounts, min_completeness=0, max_completeness=75)

print(f"Accounts with coverage gaps: {len(gaps)}")
print(f"\nBy segment:")
print(gaps["segment"].value_counts().to_string())
print(f"\nBy industry (top 5):")
print(gaps["industry"].value_counts().head().to_string())

In [ ]:
# What are the most common gaps?
from collections import Counter

# Count missing roles across all gap accounts
all_missing_roles = []
for roles in gaps["roles_missing"]:
    if isinstance(roles, list):
        all_missing_roles.extend(roles)

role_gaps = Counter(all_missing_roles)

# Count other gap types
no_vp = (~gaps["has_vp_plus"]).sum()
no_tech = (~gaps["has_technical"]).sum()
no_biz = (~gaps["has_business"]).sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Missing roles
if role_gaps:
    roles_sorted = sorted(role_gaps.items(), key=lambda x: -x[1])
    axes[0].barh([r[0] for r in roles_sorted], [r[1] for r in roles_sorted],
                 color=sns.color_palette("muted"), edgecolor="white")
    axes[0].set_xlabel("Number of Accounts Missing This Role")
    axes[0].set_title("Most Common Missing Buying Roles")
    for i, (role, count) in enumerate(roles_sorted):
        axes[0].text(count + 1, i, f"{count} ({count/len(gaps):.0%})", va="center", fontsize=9)

# Structural gaps
gap_types = {
    "No VP+ Contact": no_vp,
    "No Technical\nFunction": no_tech,
    "No Business\nFunction": no_biz,
}
bars = axes[1].bar(gap_types.keys(), gap_types.values(), color=sns.color_palette("muted")[4:],
                   edgecolor="white")
axes[1].set_ylabel("Accounts")
axes[1].set_title("Structural Gaps in Buying Groups")
for bar, val in zip(bars, gap_types.values()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f"{val} ({val/len(gaps):.0%})", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Show sample gap accounts with enrichment recommendations
print("=== Sample Enrichment Recommendations ===\n")
sample = gaps.head(10)[["account_id", "company_name", "segment", "completeness_score",
                         "contact_count", "enrichment_recommendation"]]
for _, row in sample.iterrows():
    print(f"{row['company_name']} ({row['segment']}) — Score: {row['completeness_score']:.0f}/100")
    print(f"  Contacts on deals: {row['contact_count']}")
    print(f"  Gaps: {row['enrichment_recommendation']}")
    print()

## 4. Completeness by Segment

Are certain segments more likely to have incomplete buying groups? This tells marketing where to concentrate enrichment resources.

In [ ]:
# Merge completeness with account details
active_details = active.merge(
    accounts[["account_id", "segment", "industry", "annual_revenue"]], on="account_id"
)

segment_order = ["SMB", "Mid-Market", "Enterprise"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Completeness distribution by segment
for seg in segment_order:
    subset = active_details[active_details["segment"] == seg]
    axes[0].hist(subset["completeness_score"], bins=15, alpha=0.5, label=seg)
axes[0].set_xlabel("Completeness Score")
axes[0].set_ylabel("Accounts")
axes[0].set_title("Completeness Distribution by Segment")
axes[0].legend()

# Average completeness by segment
seg_avg = active_details.groupby("segment")["completeness_score"].mean().reindex(segment_order)
bars = axes[1].bar(segment_order, seg_avg.values, color=sns.color_palette("muted"), edgecolor="white")
axes[1].set_ylabel("Average Completeness Score")
axes[1].set_title("Average Buying Group Completeness by Segment")
axes[1].set_ylim(0, 100)
for bar, val in zip(bars, seg_avg.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f"{val:.0f}", ha="center", fontsize=11)

plt.tight_layout()
plt.show()

# Sub-dimension breakdown by segment
print("Average sub-scores by segment:\n")
sub_cols = ["role_coverage_score", "seniority_mix_score", "function_diversity_score", "tech_business_score"]
seg_sub = active_details.groupby("segment")[sub_cols].mean().reindex(segment_order)
seg_sub.columns = ["Role Coverage", "Seniority Mix", "Func Diversity", "Tech+Business"]
print(seg_sub.round(1).to_string())

## 5. Pipeline Value of Enrichment

The strategic payoff: if marketing fills coverage gaps at these accounts, how much incremental pipeline could that unlock? This is the number that gets executive attention.

We conservatively assume that filling gaps increases win rate by 10 percentage points (based on the tier-level win rate differences we observed above). Estimated deal size is 2% of account annual revenue.

In [ ]:
enrichment = estimate_enrichment_pipeline(gaps, win_rate_uplift=0.10)

total_pipeline = enrichment["estimated_pipeline_uplift"].sum()
total_deal_value = enrichment["estimated_deal_value"].sum()

print(f"=== Enrichment Pipeline Estimate ===\n")
print(f"Accounts with gaps:              {len(enrichment)}")
print(f"Total estimated deal value:      ${total_deal_value:,.0f}")
print(f"Estimated pipeline uplift (10%): ${total_pipeline:,.0f}")

# Break down by segment
seg_pipeline = enrichment.groupby("segment").agg(
    accounts=("account_id", "count"),
    deal_value=("estimated_deal_value", "sum"),
    pipeline_uplift=("estimated_pipeline_uplift", "sum"),
).reindex(segment_order)

print(f"\nBy segment:")
for seg, row in seg_pipeline.iterrows():
    print(f"  {seg:15s} — {row['accounts']:3.0f} accounts, "
          f"${row['pipeline_uplift']:>12,.0f} pipeline uplift")

In [ ]:
# Visualize: pipeline uplift by segment and completeness tier
enrichment["completeness_tier"] = pd.cut(
    enrichment["completeness_score"],
    bins=[-1, 25, 50, 75],
    labels=["Low (0-25)", "Medium (25-50)", "High (50-75)"],
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pipeline uplift by segment
seg_data = enrichment.groupby("segment")["estimated_pipeline_uplift"].sum().reindex(segment_order)
bars = axes[0].bar(segment_order, seg_data.values, color=sns.color_palette("muted"), edgecolor="white")
axes[0].set_ylabel("Estimated Pipeline Uplift ($)")
axes[0].set_title("Enrichment Pipeline Uplift by Segment")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f"${x/1e6:.1f}M"))
for bar, val in zip(bars, seg_data.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + seg_data.max() * 0.02,
                 f"${val/1e6:.1f}M", ha="center", fontsize=10)

# Account count by completeness tier
tier_counts = enrichment.groupby("completeness_tier", observed=False).size()
bars2 = axes[1].bar(range(len(tier_counts)), tier_counts.values,
                    color=sns.color_palette("muted")[3:], edgecolor="white")
axes[1].set_xticks(range(len(tier_counts)))
axes[1].set_xticklabels(tier_counts.index)
axes[1].set_ylabel("Accounts")
axes[1].set_title("Gap Accounts by Completeness Tier")
for bar, val in zip(bars2, tier_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 str(val), ha="center", fontsize=10)

plt.tight_layout()
plt.show()

print("Priority: Focus enrichment on Medium (25-50) and High (50-75) accounts —")
print("they're already partially engaged, so the marginal cost of filling gaps is lowest.")

## 6. Putting It Together: Propensity + Buying Group

The most powerful view combines the propensity model scores from notebook 03 with the buying group analysis. This creates a 2x2 targeting framework:

| | **Complete Buying Group** | **Gaps Exist** |
|---|---|---|
| **High Propensity** | Close now — sales should be all over these | Highest-ROI enrichment — model says they'll buy, just need the right people |
| **Low Propensity** | Already engaged but signals are weak — nurture | Low priority — gaps AND weak signals |

In [ ]:
# Score all accounts with the propensity model from notebook 03
from src.features import build_account_features, get_modeling_dataset
from src.model import train_logistic_regression
from sklearn.model_selection import train_test_split

# Retrain on full dataset (same as notebook 03)
X_all, y_all = get_modeling_dataset(accounts, contacts, opportunities, contact_opp)
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)
lr = train_logistic_regression(X_train, y_train, cv=5)

# Score all accounts in the dataset
all_scores = lr["model"].predict_proba(X_all)[:, 1]

# Build dataset with account IDs
acct_feat = build_account_features(accounts, contacts, opportunities, contact_opp)
closed = opportunities[opportunities["stage"].isin(["Closed Won", "Closed Lost"])]
acct_target = closed.groupby("account_id")["is_won"].max().reset_index()
acct_target.columns = ["account_id", "target"]
scored = acct_feat.merge(acct_target, on="account_id", how="inner")
scored["propensity_score"] = all_scores

# Merge with completeness
combined = scored[["account_id", "propensity_score", "target"]].merge(
    completeness[["account_id", "completeness_score", "roles_missing",
                   "has_vp_plus", "has_technical", "has_business"]],
    on="account_id", how="inner"
).merge(
    accounts[["account_id", "company_name", "segment", "industry", "annual_revenue"]],
    on="account_id", how="left"
)

# Create the 2x2 framework
median_prop = combined["propensity_score"].median()
combined["propensity_tier"] = np.where(combined["propensity_score"] >= median_prop, "High Propensity", "Low Propensity")
combined["bg_tier"] = np.where(combined["completeness_score"] >= 50, "Complete/Near-Complete", "Gaps Exist")

quadrant_summary = combined.groupby(["propensity_tier", "bg_tier"]).agg(
    accounts=("account_id", "count"),
    win_rate=("target", "mean"),
    avg_revenue=("annual_revenue", "mean"),
).reset_index()

print("=== Propensity x Buying Group Framework ===\n")
for _, row in quadrant_summary.iterrows():
    print(f"{row['propensity_tier']:18s} + {row['bg_tier']:22s} — "
          f"{row['accounts']:3.0f} accounts, {row['win_rate']:.0%} win rate")

In [ ]:
# Scatter plot: propensity vs completeness, colored by outcome
fig, ax = plt.subplots(figsize=(10, 8))

won = combined[combined["target"] == 1]
lost = combined[combined["target"] == 0]

ax.scatter(lost["completeness_score"], lost["propensity_score"],
           alpha=0.4, s=30, color="#e74c3c", label=f"Lost (n={len(lost)})", zorder=2)
ax.scatter(won["completeness_score"], won["propensity_score"],
           alpha=0.6, s=40, color="#2ecc71", label=f"Won (n={len(won)})", zorder=3)

# Quadrant lines
ax.axhline(y=median_prop, color="gray", linestyle="--", alpha=0.5)
ax.axvline(x=50, color="gray", linestyle="--", alpha=0.5)

# Quadrant labels
ax.text(75, ax.get_ylim()[1] * 0.95, "Close Now", fontsize=11, fontweight="bold",
        ha="center", color="#2ecc71", alpha=0.8)
ax.text(25, ax.get_ylim()[1] * 0.95, "Enrich & Close", fontsize=11, fontweight="bold",
        ha="center", color="#DD8452", alpha=0.8)
ax.text(75, ax.get_ylim()[0] + (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.05,
        "Nurture", fontsize=11, fontweight="bold", ha="center", color="gray", alpha=0.8)
ax.text(25, ax.get_ylim()[0] + (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.05,
        "Low Priority", fontsize=11, fontweight="bold", ha="center", color="gray", alpha=0.6)

ax.set_xlabel("Buying Group Completeness Score")
ax.set_ylabel("Propensity Score")
ax.set_title("Propensity × Buying Group Completeness")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Highest-ROI enrichment targets: high propensity + gaps
enrich_targets = combined[
    (combined["propensity_tier"] == "High Propensity")
    & (combined["bg_tier"] == "Gaps Exist")
].sort_values("propensity_score", ascending=False)

print(f"=== Highest-ROI Enrichment Targets ===")
print(f"High propensity accounts with buying group gaps: {len(enrich_targets)}\n")

print(f"{'Company':<20} {'Segment':<12} {'Score':>6} {'Completeness':>13} {'Gaps'}")
print("-" * 90)
for _, row in enrich_targets.head(15).iterrows():
    missing = row["roles_missing"] if isinstance(row["roles_missing"], list) else []
    gap_str = ", ".join(missing[:3]) if missing else "structural gaps only"
    print(f"{row['company_name']:<20} {row['segment']:<12} {row['propensity_score']:>6.1%} "
          f"{row['completeness_score']:>13.0f} {gap_str}")

## Summary

### What we built
A two-layer targeting system that combines **propensity scoring** (which accounts are likely to buy) with **buying group analysis** (what's missing at each account). Together they answer the three questions marketing needs:

1. **Who should we target?** → Propensity model ranks accounts by likelihood of closing
2. **Why aren't they closing?** → Buying group gaps show what's missing (roles, seniority, functions)
3. **What should we do about it?** → Enrichment recommendations with pipeline value estimates

### Key findings

- **Buying group completeness is a strong predictor of deal outcomes** — accounts with complete groups win at significantly higher rates than those with gaps
- **The most common gaps are missing buying roles** (Champions and Decision Makers) and **lack of VP+ engagement**
- **Enterprise accounts tend to have more complete groups** (more contacts = more roles covered), but even many Enterprise accounts have structural gaps
- **The highest-ROI enrichment targets are accounts with high propensity scores but incomplete buying groups** — the model says they're likely to buy, they just need the right people at the table

### How this maps to real GTM work
In production at Atlassian, this same analysis framework identified coverage gaps across enterprise accounts that translated to **$50M in estimated pipeline** from enrichment and contact addition strategies. The synthetic data here is smaller, but the methodology is identical:
1. Score accounts on buying group completeness
2. Identify gaps by role, seniority, and function
3. Prioritize using propensity model scores
4. Estimate pipeline value to build the business case for investment